# TB Portals — 03 · Train baseline (Day-1 Gate)

Reproduce Kantipudi A2: DenseNet121 ALP regressor + cavity classifier, country-segregated test.

In [1]:
import sys, os, subprocess
REPO_DIR = "/kaggle/working/dl-project-codebase"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo",
                    "https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("repo ready at", REPO_DIR)


Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (428/428), done.


In [2]:
import sys, os, pandas as pd
REPO_DIR      = "/kaggle/working/dl-project-codebase"
KAGGLE_EXPORT = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs-full/kaggle_export_full"
WORK          = "/kaggle/working"
MANIFEST      = f'{WORK}/data/processed/tbportals_manifest.csv'
os.makedirs(f'{WORK}/data/processed', exist_ok=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv',
                   dtype={'image_id':str,'patient_id':str,'country':str})
_raw['image_path'] = _raw['image_path'].apply(
    lambda p: f"{KAGGLE_EXPORT}/{p}" if isinstance(p, str) else p
)
_raw.to_csv(MANIFEST, index=False)
print(f'Manifest ready: {len(_raw)} images -> {MANIFEST}')


Manifest ready: 16990 images -> /kaggle/working/data/processed/tbportals_manifest.csv


In [3]:
import sys, os
REPO_DIR = "/kaggle/working/dl-project-codebase"
WORK     = "/kaggle/working"
MANIFEST = f'{WORK}/data/processed/tbportals_manifest.csv'
OUT_DIR  = f'{WORK}/checkpoints/tbportals/baseline'
os.makedirs(OUT_DIR, exist_ok=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from src.training.train_tbportals_baseline import main as train_main
print('manifest:', MANIFEST)
print('out_dir: ', OUT_DIR)


manifest: /kaggle/working/data/processed/tbportals_manifest.csv
out_dir:  /kaggle/working/checkpoints/tbportals/baseline


## Step 1 — Smoke test (~2 min)
Run first to confirm no crashes.

In [4]:
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania',
    '--seeds',       '0',
    '--epochs',      '2',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--out-dir',     f'{WORK}/checkpoints/tbportals/smoke',
])


[train] device=cuda:Tesla T4
[tbportals] split held_out=Romania: train=12949 val=3218 test=823 (train/val patients 11539/2885).
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 154MB/s] 
/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Romania_seed0_K1_dann0 epoch 00 loss=0.0746 val_ALP_MAE=13.165 val_cavity_AUC=0.712 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Romania_seed0_K1_dann0 epoch 01 loss=0.0711 val_ALP_MAE=12.406 val_cavity_AUC=0.738 lambda=0.00
[train] Romania_seed0_K1_dann0 TEST -> {"held_out": "Romania", "seed": 0, "num_experts": 1, "use_dann": 0, "n_test": 823, "best_val_alp_mae": 12.406, "timika_mae": 25.283, "timika_mae_pct": 18.059, "timika_pearson": 0.556, "alp_mae": 15.211, "cavity_auc": 0.716, "cavity_f1": 0.66, "best_ckpt": "/kaggle/working/checkpoints/tbportals/smoke/Romania_seed0_K1_dann0_best.pt"}

[train] DONE 1 run(s). Results -> /kaggle/working/checkpoints/tbportals/smoke/results.csv


## Step 2 — Gate run (~90 min on P100/T4)
3 countries × seed 0 × 50 epochs. Must pass before MoE work.

In [5]:
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '0',
    '--epochs',      '50',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--amp',
    '--out-dir',     OUT_DIR,
])


[train] device=cuda:Tesla T4
[tbportals] split held_out=Romania: train=12949 val=3218 test=823 (train/val patients 11539/2885).


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Romania_seed0_K1_dann0 epoch 00 loss=0.0746 val_ALP_MAE=13.165 val_cavity_AUC=0.712 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Romania_seed0_K1_dann0 epoch 01 loss=0.0711 val_ALP_MAE=12.406 val_cavity_AUC=0.738 lambda=0.00


...

[train] Kazakhstan_seed0_K1_dann0 epoch 15 loss=0.0654 val_ALP_MAE=13.430 val_cavity_AUC=0.783 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Kazakhstan_seed0_K1_dann0 epoch 16 loss=0.0648 val_ALP_MAE=13.793 val_cavity_AUC=0.767 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Kazakhstan_seed0_K1_dann0 epoch 17 loss=0.0644 val_ALP_MAE=13.004 val_cavity_AUC=0.788 lambda=0.00
[train] Kazakhstan_seed0_K1_dann0 early stop at epoch 17 (best val ALP MAE=12.664)
[train] Kazakhstan_seed0_K1_dann0 TEST -> {"held_out": "Kazakhstan", "seed": 0, "num_experts": 1, "use_dann": 0, "n_test": 2037, "best_val_alp_mae": 12.664, "timika_mae": 22.485, "timika_mae_pct": 16.061, "timika_pearson": 0.594, "alp_mae": 12.806, "cavity_auc": 0.76, "cavity_f1": 0.645, "best_ckpt": "/kaggle/working/checkpoints/tbportals/baseline/Kazakhstan_seed0_K1_dann0_best.pt"}

[train] DONE 3 run(s). Results -> /kaggle/working/checkpoints/tbportals/baseline/results.csv


## Step 3 — Multi-seed run (seeds 1 & 2)
Run after gate passes to get mean±std for the paper.

In [6]:
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '1', '2',
    '--epochs',      '50',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--amp',
    '--out-dir',     OUT_DIR,
])


[train] device=cuda:Tesla T4
[tbportals] split held_out=Romania: train=12945 val=3222 test=823 (train/val patients 11539/2885).


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Romania_seed1_K1_dann0 epoch 00 loss=0.0746 val_ALP_MAE=21.609 val_cavity_AUC=0.722 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Romania_seed1_K1_dann0 epoch 01 loss=0.0709 val_ALP_MAE=12.426 val_cavity_AUC=0.750 lambda=0.00


...

[train] Kazakhstan_seed2_K1_dann0 epoch 32 loss=0.0597 val_ALP_MAE=12.632 val_cavity_AUC=0.800 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Kazakhstan_seed2_K1_dann0 epoch 33 loss=0.0592 val_ALP_MAE=12.470 val_cavity_AUC=0.794 lambda=0.00


/kaggle/working/dl-project-codebase/src/training/train_tbportals_baseline.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train] Kazakhstan_seed2_K1_dann0 epoch 34 loss=0.0595 val_ALP_MAE=12.716 val_cavity_AUC=0.796 lambda=0.00
[train] Kazakhstan_seed2_K1_dann0 early stop at epoch 34 (best val ALP MAE=12.180)
[train] Kazakhstan_seed2_K1_dann0 TEST -> {"held_out": "Kazakhstan", "seed": 2, "num_experts": 1, "use_dann": 0, "n_test": 2037, "best_val_alp_mae": 12.18, "timika_mae": 24.401, "timika_mae_pct": 17.429, "timika_pearson": 0.525, "alp_mae": 13.717, "cavity_auc": 0.755, "cavity_f1": 0.645, "best_ckpt": "/kaggle/working/checkpoints/tbportals/baseline/Kazakhstan_seed2_K1_dann0_best.pt"}

[train] DONE 6 run(s). Results -> /kaggle/working/checkpoints/tbportals/baseline/results.csv


In [7]:
# Kantipudi et al. (JIIM 2024), best model A2, country-segregated test.
KANTIPUDI_A2: dict[str, dict[str, float]] = {
    "Romania": {"timika_mae": 18.70, "timika_mae_pct": 13.36, "timika_pearson": 0.70,
                "alp_mae": 11.86, "cavity_auc": 0.80, "cavity_f1": 0.81},
    "Moldova": {"timika_mae": 18.85, "timika_mae_pct": 13.46, "timika_pearson": 0.84,
                "alp_mae": 16.24, "cavity_auc": 0.88, "cavity_f1": 0.71},
    "Kazakhstan": {"timika_mae": 19.62, "timika_mae_pct": 14.01, "timika_pearson": 0.70,
                   "alp_mae": 12.16, "cavity_auc": 0.85, "cavity_f1": 0.72},
}

# Verifying if we were ablew to replicate Kantipudi et al. results

In [8]:
res = pd.read_csv(f'/kaggle/working/checkpoints/tbportals/baseline/results.csv')
passed = []
for _, row in res.iterrows():
    gate = KANTIPUDI_A2.get(row['held_out'], {})
    ok = all(row.get(k, float('inf')) <= v for k, v in gate.items())
    status = 'PASS' if ok else 'FAIL'
    passed.append(ok)
    print(f"{status}  {row['held_out']} seed={row['seed']}  "
          f"ALP_MAE={row.get('alp_mae',float('nan')):.3f}  "
          f"cavity_AUC={row.get('cavity_auc',float('nan')):.3f}")
print()
print('Overall:', 'ALL PASS' if all(passed) else f'{sum(passed)}/{len(passed)} passed')


FAIL  Romania seed=0  ALP_MAE=13.627  cavity_AUC=0.733
FAIL  Moldova seed=0  ALP_MAE=22.145  cavity_AUC=0.846
FAIL  Kazakhstan seed=0  ALP_MAE=12.806  cavity_AUC=0.760
FAIL  Romania seed=1  ALP_MAE=13.644  cavity_AUC=0.751
FAIL  Romania seed=2  ALP_MAE=14.481  cavity_AUC=0.734
FAIL  Moldova seed=1  ALP_MAE=25.807  cavity_AUC=0.816
FAIL  Moldova seed=2  ALP_MAE=28.211  cavity_AUC=0.782
FAIL  Kazakhstan seed=1  ALP_MAE=12.344  cavity_AUC=0.764
FAIL  Kazakhstan seed=2  ALP_MAE=13.717  cavity_AUC=0.755

Overall: 0/9 passed


In [9]:
!zip -j /kaggle/working/results.zip /kaggle/working/checkpoints/tbportals/baseline/results.csv

  adding: results.csv (deflated 67%)
